In [1]:
import pandas as pd
import sqlite3

dim_users = pd.read_csv("/Users/aishwarya/Desktop/Projects/Product-Funnel-Retention-Analysis-for-a-Wellness-App/data/dim_users.csv")
fact_events = pd.read_csv("/Users/aishwarya/Desktop/Projects/Product-Funnel-Retention-Analysis-for-a-Wellness-App/data/fact_events.csv")

fact_events["event_ts"] = pd.to_datetime(fact_events["event_ts"])
fact_events["event_date"] = pd.to_datetime(fact_events["event_date"])

dim_users.shape, fact_events.shape
fact_events["event_name"].value_counts()

session_started         18445
session_completed       18445
app_install             10000
sign_up                 10000
onboarding_completed     5125
subscription_started      909
Name: event_name, dtype: int64

In [2]:
conn = sqlite3.connect("product_analytics.db")

dim_users.to_sql("dim_users", conn, if_exists="replace", index=False)
fact_events.to_sql("fact_events", conn, if_exists="replace", index=False)

In [4]:
pd.read_sql("""
SELECT *
FROM fact_events
limit 20
""", conn)

,event_id,user_id,event_name,event_ts,event_date,session_id
0,afa26443-c71a-4e27-8ee9-6235ea790ab6,0000142a-fb28-440d-8be1-30c131aad266,app_install,2025-01-30 22:33:25,2025-01-30 00:00:00,None
1,e7bd5af9-6d6b-45ce-a162-bca78d30bfec,0000142a-fb28-440d-8be1-30c131aad266,sign_up,2025-01-30 22:38:47,2025-01-30 00:00:00,None
2,61010338-a5ac-4c40-acda-faaca62caa49,0000142a-fb28-440d-8be1-30c131aad266,onboarding_completed,2025-01-30 22:49:47,2025-01-30 00:00:00,None
3,14daccbc-a999-4101-8ae0-dd975ee12a7e,0000142a-fb28-440d-8be1-30c131aad266,session_started,2025-02-25 02:41:26,2025-02-25 00:00:00,804155c4-ddb6-4cb9-948c-7778d895b083
4,bc54b8d1-9ae8-4e54-b43d-a7217bb19632,0000142a-fb28-440d-8be1-30c131aad266,session_completed,2025-02-25 02:46:26,2025-02-25 00:00:00,804155c4-ddb6-4cb9-948c-7778d895b083
5,30853cff-8fbf-48ba-8dc5-1d8f8917baa0,0007eaaa-bb53-460e-9441-4cf87e41f8db,app_install,2025-01-13 12:31:56,2025-01-13 00:00:00,None
6,f9ea3843-c522-4cee-9389-c1c00f0c866e,0007eaaa-bb53-460e-9441-4cf87e41f8db,sign_up,2025-01-13 12:39:11,2025-01-13 00:00:00,None
7,751b15cd-7a2d-4e69-a72d-8d3502784c69,0007eaaa-bb53-460e-9441-4cf87e41f8db,onboarding_completed,2025-01-13 12:44:11,2025-01-13 00:00:00,None
8,eecd233d-1c68-4ae6-a08b-cd9dfa67235b,0007eaaa-bb53-460e-9441-4cf87e41f8db,session_started,2025-01-18 17:34:08,2025-01-18 00:00:00,a5bb2a5c-7eb6-4d1d-ac9c-4a031fcb978c
9,ae6de62a-3bee-4747-87a6-6b0e04d4606f,0007eaaa-bb53-460e-9441-4cf87e41f8db,session_completed,2025-01-18 17:54:08,2025-01-18 00:00:00,a5bb2a5c-7eb6-4d1d-ac9c-4a031fcb978c


In [3]:
pd.read_sql("""
SELECT DISTINCT event_name
FROM fact_events
""", conn)

,event_name
0,app_install
1,sign_up
2,onboarding_completed
3,session_started
4,session_completed
5,subscription_started


In [7]:
pd.read_sql("""
SELECT event_name, count(distinct user_id) as no_users
FROM fact_events
group by event_name
""", conn)

,event_name,no_users
0,app_install,10000
1,onboarding_completed,5125
2,session_completed,4755
3,session_started,4755
4,sign_up,10000
5,subscription_started,909


app_install --> Did the users discover the app?
sign_up --> Did the users commit to it?
onboarding_complete --> Did they understnad the product?
session_started --> Did they actually use it?
subscription_started --> Did they pay?

In [19]:
pd.read_sql("""
SELECT
event_name,
count(distinct user_id) as no_users
FROM fact_events
where event_name in ('app_install','onboarding_completed','session_started','sign_up','subscription_started')
group by event_name
order by 
case 
    when event_name = 'app_install' then 1
    when event_name = 'sign_up' then 2
    when event_name = 'onboarding_completed' then 3
    when event_name = 'session_started' then 4
    when event_name = 'subscription_started' then 5
end
""", conn)

,event_name,no_users
0,app_install,10000
1,sign_up,10000
2,onboarding_completed,5125
3,session_started,4755
4,subscription_started,909


In [28]:
pd.read_sql("""
with funnel_steps as (
    SELECT
    event_name as step,
    count(distinct user_id) as no_users,
    case 
        when event_name = 'app_install' then 1
        when event_name = 'sign_up' then 2
        when event_name = 'onboarding_completed' then 3
        when event_name = 'session_started' then 4
        when event_name = 'subscription_started' then 5
    end as step_no
    FROM fact_events
    where event_name in ('app_install','onboarding_completed','session_started','sign_up','subscription_started')
    group by event_name
    order by step_no
    ) select step, no_users,
    lag(no_users,1) over (order by step_no) as prev_usercount
    from funnel_steps;
""", conn)

,step,no_users,prev_usercount
0,app_install,10000,NaN
1,sign_up,10000,10000.0
2,onboarding_completed,5125,10000.0
3,session_started,4755,5125.0
4,subscription_started,909,4755.0


In [45]:
pd.read_sql("""
with funnel_steps as (
    SELECT
    event_name as step,
    count(distinct user_id) as no_users,
    case 
        when event_name = 'app_install' then 1
        when event_name = 'sign_up' then 2
        when event_name = 'onboarding_completed' then 3
        when event_name = 'session_started' then 4
        when event_name = 'subscription_started' then 5
    end as step_no
    FROM fact_events
    where event_name in ('app_install','onboarding_completed','session_started','sign_up','subscription_started')
    group by event_name
    order by step_no
    ),
    previous_counts as (
    select step, no_users, step_no,
    lag(no_users,1) over (order by step_no) as prev_usercount
    from funnel_steps
    )
    select *,
    case
    when prev_usercount is null then null
    when prev_usercount is not null then cast(no_users as REAL)/prev_usercount
    end as conversion
    from previous_counts
    order by step_no
    ;
""", conn)

,step,no_users,step_no,prev_usercount,conversion
0,app_install,10000,1,NaN,NaN
1,sign_up,10000,2,10000.0,1.000000
2,onboarding_completed,5125,3,10000.0,0.512500
3,session_started,4755,4,5125.0,0.927805
4,subscription_started,909,5,4755.0,0.191167


In [58]:
pd.read_sql("""
with funnel_steps as (
    SELECT
    event_name as step,
    count(distinct user_id) as no_users,
    case 
        when event_name = 'app_install' then 1
        when event_name = 'sign_up' then 2
        when event_name = 'onboarding_completed' then 3
        when event_name = 'session_started' then 4
        when event_name = 'subscription_started' then 5
    end as step_no
    FROM fact_events
    where event_name in ('app_install','onboarding_completed','session_started','sign_up','subscription_started')
    group by event_name
    order by step_no
    ),
    previous_counts as (
    select step, no_users, step_no,
    lag(no_users,1) over (order by step_no) as prev_usercount
    from funnel_steps
    ),
    conversion as (
    select *,
    case
    when prev_usercount is null then null
    when prev_usercount is not null then cast(no_users as REAL)/prev_usercount
    end as conversion
    from previous_counts
    )
    select *,
    cast(no_users as real) / first_value(no_users) over (order by step_no) as overall_conversion
    from conversion
    order by step_no;
""", conn)

,step,no_users,step_no,prev_usercount,conversion,overall_conversion
0,app_install,10000,1,NaN,NaN,1.0000
1,sign_up,10000,2,10000.0,1.000000,1.0000
2,onboarding_completed,5125,3,10000.0,0.512500,0.5125
3,session_started,4755,4,5125.0,0.927805,0.4755
4,subscription_started,909,5,4755.0,0.191167,0.0909


### Assumptions from the funnel table
## - The prev_usercount column allows us to analyze two things,
     1. It tells us the conversion of people from each step. For example, all the users who installed the app also signed up for the app but not everyone who signed up completed the on boarding of the app
     2. It allows us to analyze the overall conversion meaning compare everyone who installed the app convert to the next step.
## - Looking at the numbers we can make the below assumptions
     1. Everyone who installed the app also signed up to the app
     2. Only 51% of people who signed up to the app actually on-boarded it.
     3. There is a significant 19% drop from people who started a session who proceeded to a subscription.
     4. We also learn from the numbers that out of all the people who installed the app only 47% of the people started a session and only 9% started a subscription.
     5. This tells us that we need to improve or make changes for better user experience to get people to subscribe more after a session


**This could indicate that users do not perceive enough value during early sessions, suggesting a need to improve feature discovery or introduce stronger paywall triggers.**

In [92]:
funnel_df = pd.read_sql("""
with funnel_steps as (
    SELECT
    event_name as step,
    count(distinct user_id) as no_users,
    case 
        when event_name = 'app_install' then 1
        when event_name = 'sign_up' then 2
        when event_name = 'onboarding_completed' then 3
        when event_name = 'session_started' then 4
        when event_name = 'subscription_started' then 5
    end as step_no
    FROM fact_events
    where event_name in ('app_install','onboarding_completed','session_started','sign_up','subscription_started')
    group by event_name
    order by step_no
    ),
    previous_counts as (
    select step, no_users, step_no,
    lag(no_users,1) over (order by step_no) as prev_usercount
    from funnel_steps
    ),
    conversion as (
    select *,
    case
    when prev_usercount is null then null
    when prev_usercount is not null then cast(no_users as REAL)/prev_usercount
    end as conversion
    from previous_counts
    )
    select *,
    cast(no_users as real) / first_value(no_users) over (order by step_no) as overall_conversion
    from conversion
    order by step_no;
""", conn)

In [93]:
funnel_df.head()

,step,no_users,step_no,prev_usercount,conversion,overall_conversion
0,app_install,10000,1,NaN,NaN,1.0000
1,sign_up,10000,2,10000.0,1.000000,1.0000
2,onboarding_completed,5125,3,10000.0,0.512500,0.5125
3,session_started,4755,4,5125.0,0.927805,0.4755
4,subscription_started,909,5,4755.0,0.191167,0.0909


In [94]:
funnel_df.shape

(5, 6)

In [95]:
funnel_df.to_csv("/Users/aishwarya/Desktop/Projects/Product-Funnel-Retention-Analysis-for-a-Wellness-App/data/funnel_metrics.csv", index=False)

In [61]:
pd.read_sql("""
SELECT *
FROM fact_events
limit 5
""", conn)

,event_id,user_id,event_name,event_ts,event_date,session_id
0,afa26443-c71a-4e27-8ee9-6235ea790ab6,0000142a-fb28-440d-8be1-30c131aad266,app_install,2025-01-30 22:33:25,2025-01-30 00:00:00,None
1,e7bd5af9-6d6b-45ce-a162-bca78d30bfec,0000142a-fb28-440d-8be1-30c131aad266,sign_up,2025-01-30 22:38:47,2025-01-30 00:00:00,None
2,61010338-a5ac-4c40-acda-faaca62caa49,0000142a-fb28-440d-8be1-30c131aad266,onboarding_completed,2025-01-30 22:49:47,2025-01-30 00:00:00,None
3,14daccbc-a999-4101-8ae0-dd975ee12a7e,0000142a-fb28-440d-8be1-30c131aad266,session_started,2025-02-25 02:41:26,2025-02-25 00:00:00,804155c4-ddb6-4cb9-948c-7778d895b083
4,bc54b8d1-9ae8-4e54-b43d-a7217bb19632,0000142a-fb28-440d-8be1-30c131aad266,session_completed,2025-02-25 02:46:26,2025-02-25 00:00:00,804155c4-ddb6-4cb9-948c-7778d895b083


| user_id | signup_date | event_date | days_since_signup |

In [62]:
pd.read_sql("""
SELECT *
FROM dim_users
limit 5
""", conn)

,user_id,signup_ts,platform,persona,country
0,27df2322-322b-44d5-a6f9-00e75037da2a,2025-01-06 09:42:36,iOS,casual,Germany
1,242b3c5d-f9be-4fac-82a3-1103a9009a83,2025-01-02 05:08:22,Android,one-time,Germany
2,8254d52c-8798-4319-91c2-45eba48fbdbd,2025-01-14 08:25:56,Web,one-time,Nepal
3,6713e50b-97e0-4fc1-a901-ae9d591b2d9d,2025-01-12 21:19:10,Android,casual,Australia
4,9f8c5ec5-3cbb-46b7-888c-7503175d1316,2025-01-11 20:03:33,Web,power,USA


In [72]:
pd.read_sql("""
SELECT f.user_id,
date(d.signup_ts) as signup_date,
date(f.event_date) as event_date,
cast(julianday(date(f.event_date)) - julianday(date(d.signup_ts)) as integer) as days_since_signup
from fact_events f
inner join dim_users d
on f.user_id = d.user_id
where f.event_name = "session_started"
""", conn)

,user_id,signup_date,event_date,days_since_signup
0,0000142a-fb28-440d-8be1-30c131aad266,2025-01-30,2025-02-25,26
1,0007eaaa-bb53-460e-9441-4cf87e41f8db,2025-01-13,2025-01-18,5
2,0007eaaa-bb53-460e-9441-4cf87e41f8db,2025-01-13,2025-01-25,12
3,0007eaaa-bb53-460e-9441-4cf87e41f8db,2025-01-13,2025-01-28,15
4,0007eaaa-bb53-460e-9441-4cf87e41f8db,2025-01-13,2025-02-12,30
...,...,...,...,...
18440,ffba1db5-cd3b-4b5e-b6a3-b53929383363,2025-01-19,2025-02-09,21
18441,fff27cf6-3ad9-4868-8a5a-66f6aeddbf5a,2025-01-09,2025-01-17,8
18442,fff27cf6-3ad9-4868-8a5a-66f6aeddbf5a,2025-01-09,2025-01-19,10
18443,fff27cf6-3ad9-4868-8a5a-66f6aeddbf5a,2025-01-09,2025-01-29,20


In [91]:
pd.read_sql("""
with day_sincesignup as (
    select
        f.user_id,
        cast(julianday(date(f.event_date)) - julianday(date(d.signup_ts)) as integer) as days_since_signup
    from fact_events f
    inner join dim_users d
        on f.user_id = d.user_id
    where f.event_name = 'session_started'
),
retention_counts as (
    select
        count(distinct case when days_since_signup = 1 then user_id end) as d1_users,
        count(distinct case when days_since_signup <= 7 then user_id end) as d7_users,
        count(distinct case when days_since_signup <= 30 then user_id end) as d30_users
    from day_sincesignup
),
total_users as (
    select count(distinct user_id) as total_users
    from dim_users
)
select
    t.total_users,
    r.d1_users,
    r.d7_users,
    r.d30_users,
    cast(r.d1_users as real) / t.total_users as d1_retention,
    cast(r.d7_users as real) / t.total_users as d7_retention,
    cast(r.d30_users as real) / t.total_users as d30_retention
from retention_counts r
cross join total_users t;
""", conn)

,total_users,d1_users,d7_users,d30_users,d1_retention,d7_retention,d30_retention
0,10000,546,2896,4751,0.0546,0.2896,0.4751


In [96]:
retention_df = pd.read_sql("""
with day_sincesignup as (
    select
        f.user_id,
        cast(julianday(date(f.event_date)) - julianday(date(d.signup_ts)) as integer) as days_since_signup
    from fact_events f
    inner join dim_users d
        on f.user_id = d.user_id
    where f.event_name = 'session_started'
),
retention_counts as (
    select
        count(distinct case when days_since_signup = 1 then user_id end) as d1_users,
        count(distinct case when days_since_signup <= 7 then user_id end) as d7_users,
        count(distinct case when days_since_signup <= 30 then user_id end) as d30_users
    from day_sincesignup
),
total_users as (
    select count(distinct user_id) as total_users
    from dim_users
)
select
    t.total_users,
    r.d1_users,
    r.d7_users,
    r.d30_users,
    cast(r.d1_users as real) / t.total_users as d1_retention,
    cast(r.d7_users as real) / t.total_users as d7_retention,
    cast(r.d30_users as real) / t.total_users as d30_retention
from retention_counts r
cross join total_users t;
""", conn)

In [97]:
retention_df.head()

,total_users,d1_users,d7_users,d30_users,d1_retention,d7_retention,d30_retention
0,10000,546,2896,4751,0.0546,0.2896,0.4751


In [98]:
retention_df.shape

(1, 7)

In [99]:
retention_df.to_csv("/Users/aishwarya/Desktop/Projects/Product-Funnel-Retention-Analysis-for-a-Wellness-App/data/retention_metrics.csv", index=False)

In [100]:
import os
os.listdir("/Users/aishwarya/Desktop/Projects/Product-Funnel-Retention-Analysis-for-a-Wellness-App/data/")

['funnel_metrics.csv',
 '.DS_Store',
 'fact_events.csv',
 'dim_users.csv',
 'generate_events.py',
 'generate_fact_events.ipynb',
 'funnel_retention_analysis.ipynb',
 'fact_events_sample.csv',
 'product_analytics.db',
 'generate_events.ipynb',
 'retention_metrics.csv',
 'dim_users_sample.csv',
 '.ipynb_checkpoints']